## परिचय

इस पाठ में शामिल होंगे:
- फंक्शन कॉलिंग क्या है और इसके उपयोग के मामले
- Azure OpenAI का उपयोग करके फंक्शन कॉल कैसे बनाएं
- एक एप्लिकेशन में फंक्शन कॉल को कैसे इंटीग्रेट करें

## सीखने के लक्ष्य

इस पाठ को पूरा करने के बाद आप जानेंगे और समझेंगे:

- फंक्शन कॉलिंग का उपयोग करने का उद्देश्य
- Azure Open AI सेवा का उपयोग करके फंक्शन कॉल सेटअप करना
- अपने एप्लिकेशन के उपयोग के मामले के लिए प्रभावी फंक्शन कॉल डिज़ाइन करना


## फ़ंक्शन कॉल्स को समझना 

इस पाठ के लिए, हम अपनी शिक्षा स्टार्टअप के लिए एक फीचर बनाना चाहते हैं जो उपयोगकर्ताओं को तकनीकी कोर्स खोजने के लिए चैटबॉट का उपयोग करने की अनुमति देता है। हम ऐसे कोर्स की सिफारिश करेंगे जो उनके कौशल स्तर, वर्तमान भूमिका और रुचि की तकनीक के अनुकूल हों। 

इसे पूरा करने के लिए हम निम्न संयोजन का उपयोग करेंगे: 
 - उपयोगकर्ता के लिए चैट अनुभव बनाने के लिए `Azure Open AI`
 - उपयोगकर्ता के अनुरोध के आधार पर कोर्स खोजने में मदद करने के लिए `Microsoft Learn Catalog API`
 - उपयोगकर्ता के प्रश्न को लेकर API अनुरोध करने के लिए `Function Calling`

शुरू करने के लिए, आइए देखें कि हम सबसे पहले फ़ंक्शन कॉलिंग का उपयोग क्यों करना चाहेंगे: 


### फंक्शन कॉलिंग क्यों

यदि आपने इस कोर्स का कोई अन्य पाठ पूरा किया है, तो आप शायद बड़े भाषा मॉडल (LLMs) का उपयोग करने की शक्ति को समझते हैं। उम्मीद है कि आप उनकी कुछ सीमाओं को भी देख सकते हैं।

फंक्शन कॉलिंग Azure Open AI सेवा की एक विशेषता है जो निम्नलिखित सीमाओं को पार करने के लिए है:
1) सुसंगत उत्तर प्रारूप
2) चैट संदर्भ में किसी एप्लिकेशन के अन्य स्रोतों के डेटा का उपयोग करने की क्षमता

फंक्शन कॉलिंग से पहले, LLM से प्राप्त उत्तर असंरचित और असंगत थे। डेवलपर्स को प्रत्येक प्रकार के उत्तर को संभालने के लिए जटिल सत्यापन कोड लिखना पड़ता था।

उपयोगकर्ता ऐसे उत्तर नहीं प्राप्त कर सकते थे जैसे "स्टॉकहोम में वर्तमान मौसम क्या है?"। ऐसा इसलिए था क्योंकि मॉडल उस समय तक के डेटा तक सीमित थे जिस समय उन्हें प्रशिक्षित किया गया था।

नीचे दिए गए उदाहरण पर गौर करें जो इस समस्या को दर्शाता है:

मान लीजिए हम छात्रों के डेटा का एक डेटाबेस बनाना चाहते हैं ताकि हम उन्हें सही कोर्स सुझा सकें। नीचे हमारे पास दो छात्रों के विवरण हैं जो डेटा के संदर्भ में बहुत समान हैं।


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


हम इसे डेटा पार्स करने के लिए एक LLM को भेजना चाहते हैं। इसका उपयोग बाद में हमारे एप्लिकेशन में इस जानकारी को एक API को भेजने या इसे एक डेटाबेस में स्टोर करने के लिए किया जा सकता है।

आइए दो एक सामान प्रॉम्प्ट बनाते हैं जिनमें हम LLM को यह निर्देश देते हैं कि हमें किस जानकारी में रुचि है: 


हम इसे हमारे उत्पाद के लिए महत्वपूर्ण भागों को पार्स करने के लिए एक LLM को भेजना चाहते हैं। ताकि हम LLM को निर्देश देने के लिए दो समान प्रॉम्प्ट बना सकें: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


इन दो प्रांप्ट्स को बनाने के बाद, हम उन्हें `client.responses.create` का उपयोग करके LLM को भेजेंगे। हम प्रांप्ट को `input` वेरिएबल में संग्रहित करते हैं और भूमिका को `user` असाइन करते हैं। यह एक उपयोगकर्ता द्वारा चैटबोट को लिखा गया संदेश बनाने के लिए है।



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

अब हम दोनों अनुरोधों को LLM को भेज सकते हैं और प्राप्त प्रतिक्रिया की जांच कर सकते हैं। 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



भले ही प्रॉम्प्ट एक जैसे हों और विवरण समान हों, हम `Grades` प्रापर्टी के विभिन्न स्वरूप प्राप्त कर सकते हैं।

यदि आप ऊपर का सेल कई बार चलाते हैं, तो स्वरूप `3.7` या `3.7 GPA` हो सकता है।

ऐसा इसलिए है क्योंकि LLM लिखित प्रॉम्प्ट के रूप में असंरचित डेटा लेता है और असंरचित डेटा वापस करता है। हमें एक संरचित स्वरूप चाहिए ताकि हम जानते हों कि इस डेटा को संग्रहीत या उपयोग करते समय क्या उम्मीद करनी है।

फ़ंक्शन कॉलिंग का उपयोग करके, हम सुनिश्चित कर सकते हैं कि हमें संरचित डेटा वापस मिले। फ़ंक्शन कॉलिंग का उपयोग करते समय, LLM वास्तव में किसी भी फ़ंक्शन को कॉल या चलाता नहीं है। इसके स्थान पर, हम LLM के लिए उसकी प्रतिक्रियाओं का पालन करने के लिए एक संरचना बनाते हैं। फिर हम उन संरचित प्रतिक्रियाओं का उपयोग करके जानते हैं कि हमारे अनुप्रयोगों में कौन सा फ़ंक्शन चलाना है।
 


![फंक्शन कॉलिंग फ्लो आरेख](../../../../translated_images/hi/Function-Flow.083875364af4f4bb.webp)


हम तब उस फ़ंक्शन से वापस आए हुए परिणाम को लेकर इसे फिर से LLM को भेज सकते हैं। LLM फिर स्वाभाविक भाषा का उपयोग करके उपयोगकर्ता के प्रश्न का उत्तर देगा। 


### फ़ंक्शन कॉल का उपयोग करने के लिए उपयोग के मामले

**बाहरी टूल्स को कॉल करना**
चैटबॉट उपयोगकर्ताओं के प्रश्नों के उत्तर प्रदान करने में उत्कृष्ट होते हैं। फ़ंक्शन कॉलिंग का उपयोग करके, चैटबॉट उपयोगकर्ता के संदेशों का उपयोग कुछ कार्यों को पूरा करने के लिए कर सकते हैं। उदाहरण के लिए, एक छात्र चैटबॉट से पूछ सकता है "मेरे प्रशिक्षक को ईमेल भेजो जिसमें कहा गया हो कि मुझे इस विषय में अधिक सहायता चाहिए"। यह `send_email(to: string, body: string)` फ़ंक्शन कॉल कर सकता है।


**API या डेटाबेस प्रश्न बनाना**
उपयोगकर्ता प्राकृतिक भाषा का उपयोग करके जानकारी खोज सकते हैं जो एक स्वरूपित क्वेरी या API अनुरोध में परिवर्तित हो जाती है। इसका एक उदाहरण एक शिक्षक हो सकता है जो पूछता है "वे छात्र कौन हैं जिन्होंने आखिरी असाइनमेंट पूरा किया" जो `get_completed(student_name: string, assignment: int, current_status: string)` नामक एक फ़ंक्शन को कॉल कर सकता है।


**संरचित डेटा बनाना**
उपयोगकर्ता एक टेक्स्ट या CSV ब्लॉक को लेकर LLM से उसमें से महत्वपूर्ण जानकारी निकाल सकते हैं। उदाहरण के लिए, एक छात्र शांति समझौतों के बारे में विकिपीडिया लेख को AI फ्लैश कार्ड बनाने के लिए परिवर्तित कर सकता है। यह `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` नामक एक फ़ंक्शन का उपयोग करके किया जा सकता है।


## 2. अपनी पहली फ़ंक्शन कॉल बनाना 

फ़ंक्शन कॉल बनाने की प्रक्रिया में 3 मुख्य चरण शामिल हैं: 
1. अपनी फ़ंक्शन सूची और उपयोगकर्ता संदेश के साथ चैट कम्प्लीशंस API को कॉल करना 
2. एक्शन करने के लिए मॉडल की प्रतिक्रिया पढ़ना, जैसे कि फ़ंक्शन या API कॉल को निष्पादित करना 
3. अपने फ़ंक्शन से प्राप्त प्रतिक्रिया के साथ चैट कम्प्लीशंस API को फिर से कॉल करना ताकि उस जानकारी का उपयोग करके उपयोगकर्ता को प्रतिक्रिया बनाई जा सके। 


![एक फ़ंक्शन कॉल का प्रवाह](../../../../translated_images/hi/LLM-Flow.3285ed8caf4796d7.webp)


### फ़ंक्शन कॉल के तत्व 

#### उपयोगकर्ता इनपुट 

पहला कदम एक उपयोगकर्ता संदेश बनाना है। इसे टेक्स्ट इनपुट का मान लेकर गतिशील रूप से असाइन किया जा सकता है या आप यहां एक मान असाइन कर सकते हैं। यदि यह आपकी पहली बार Chat Completions API के साथ काम कर रहे हैं, तो हमें संदेश का `role` और `content` परिभाषित करना होगा। 

`role` या तो `system` (नियम बनाना), `assistant` (मॉडल) या `user` (अंतिम उपयोगकर्ता) हो सकता है। फ़ंक्शन कॉलिंग के लिए, हम इसे `user` और एक उदाहरण प्रश्न के रूप में असाइन करेंगे। 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### फ़ंक्शन बनाना। 

अब हम एक फ़ंक्शन और उस फ़ंक्शन के पैरामीटर को परिभाषित करेंगे। हम यहाँ केवल एक फ़ंक्शन का उपयोग करेंगे जिसे `search_courses` कहा जाता है, लेकिन आप कई फ़ंक्शन बना सकते हैं।

**महत्वपूर्ण** : फ़ंक्शन सिस्टम संदेश में LLM के लिए शामिल होते हैं और आपके पास उपलब्ध टोकन की मात्रा में शामिल होंगे। 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**परिभाषाएँ** 

`name` - उस फ़ंक्शन का नाम जिसे हम कॉल करना चाहते हैं। 

`description` - यह फ़ंक्शन कैसे काम करता है इसका विवरण है। यहाँ स्पष्ट और विशिष्ट होना महत्वपूर्ण है 

`parameters` - उन मानों और प्रारूपों की सूची जिन्हें आप चाहते हैं कि मॉडल अपनी प्रतिक्रिया में उत्पन्न करे 


`type` - उन गुणों का डेटा प्रकार जिनमें संग्रह किया जाएगा 

`properties` - उन विशिष्ट मानों की सूची जो मॉडल अपनी प्रतिक्रिया के लिए उपयोग करेगा 


`name` - उस गुण का नाम जिसे मॉडल अपने स्वरूपित प्रतिक्रिया में उपयोग करेगा 

`type` - इस गुण का डेटा प्रकार 

`description` - विशेष गुण का विवरण 


**वैकल्पिक**

`required` - फ़ंक्शन कॉल को पूरा करने के लिए आवश्यक गुण 


### फ़ंक्शन कॉल करना  
एक फ़ंक्शन को परिभाषित करने के बाद, हमें इसे Chat Completion API के कॉल में शामिल करने की आवश्यकता है। हम यह अनुरोध में `functions` जोड़कर करते हैं। इस मामले में `functions=functions`।  

`function_call` को `auto` सेट करने का विकल्प भी है। इसका मतलब है कि हम उपयोगकर्ता संदेश के आधार पर कौन सा फ़ंक्शन कॉल किया जाना चाहिए, यह तय करने के लिए LLM को स्वतंत्र छोड़ देंगे, बजाय इसके कि हम इसे स्वयं असाइन करें।  


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


अब चलिए प्रतिक्रिया को देखते हैं और देखते हैं कि यह कैसे स्वरूपित है: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

आप देख सकते हैं कि फ़ंक्शन का नाम कॉल किया गया है और उपयोगकर्ता संदेश से, LLM फ़ंक्शन के तर्कों को फिट करने के लिए डेटा खोज पाया। 


## 3. एक एप्लिकेशन में फ़ंक्शन कॉल्स को एकीकृत करना। 


जब हमने LLM से प्राप्त स्वरूपित प्रतिक्रिया का परीक्षण कर लिया है, तो अब हम इसे एक एप्लिकेशन में एकीकृत कर सकते हैं। 

### प्रवाह प्रबंधन 

इसे हमारे एप्लिकेशन में एकीकृत करने के लिए, आइए निम्नलिखित कदम उठाएं: 

सबसे पहले, Open AI सेवाओं को कॉल करें और संदेश को एक वेरिएबल `response_message` में संग्रहित करें। 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


अब हम वह फ़ंक्शन परिभाषित करेंगे जो Microsoft Learn API को कॉल कर पाठ्यक्रमों की सूची प्राप्त करेगा: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



एक सर्वोत्तम अभ्यास के रूप में, हम देखेंगे कि क्या मॉडल किसी फंक्शन को कॉल करना चाहता है। उसके बाद, हम उपलब्ध फ़ंक्शनों में से एक बनाएंगे और उसे उस फ़ंक्शन से मिलाएंगे जिसे कॉल किया जा रहा है।
हम फिर फ़ंक्शन के तर्क लेंगे और उन्हें LLM के तर्कों से मैप करेंगे।

अंत में, हम फ़ंक्शन कॉल संदेश और `search_courses` संदेश द्वारा लौटाए गए मानों को जोड़ेंगे। इससे LLM के पास वह सारी जानकारी होगी जिसकी इसे उपयोगकर्ता को प्राकृतिक भाषा में जवाब देने के लिए आवश्यकता है।
उपयोगकर्ता को प्राकृतिक भाषा में जवाब देने के लिए। 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


अब हम अपडेट किया गया संदेश LLM को भेजेंगे ताकि हमें API JSON स्वरूपित प्रतिक्रिया के बजाय प्राकृतिक भाषा में प्रतिक्रिया प्राप्त हो सके। 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## कोड चुनौती 

शानदार काम! Azure Open AI फंक्शन कॉलिंग की अपनी सीख जारी रखने के लिए आप बना सकते हैं: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - फंक्शन के और भी पैरामीटर जो शिक्षार्थियों को और पाठ्यक्रम खोजने में मदद कर सकते हैं। आप उपलब्ध API पैरामीटर यहाँ पा सकते हैं: 
 - एक अन्य फंक्शन कॉल बनाएं जो शिक्षार्थी से उनकी मातृ भाषा जैसी अतिरिक्त जानकारी लेता है 
 - त्रुटि हैंडलिंग बनाएं जब फंक्शन कॉल और/या API कॉल कोई उपयुक्त पाठ्यक्रम वापस नहीं करता है 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
